# Phase 3-C：NLP 自動題材發現

本 notebook 展示如何從廣域中文財經新聞（無預設關鍵字）中，
用 BERTopic 自動發現市場正在討論的投資題材。

**流程**
1. 載入廣域新聞（`news_broad.py` 採集的結果）
2. BERTopic 聚類（jieba 斷詞 + Sentence Transformer）
3. 人工審閱候選題材
4. 輸出 `topics.yaml` 草稿條目

In [ ]:
import sys
import logging
from pathlib import Path

# 確保專案根目錄在 Python path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 設定 log 等級：只顯示我們自己的 INFO
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)s %(name)s — %(message)s',
                    datefmt='%H:%M:%S')
for noisy in ['sentence_transformers','bertopic','hdbscan','umap',
              'numba','jieba','matplotlib']:
    logging.getLogger(noisy).setLevel(logging.WARNING)

print('環境準備完成')

## 1. 廣域新聞資料概覽

In [ ]:
from src.collectors.news_broad import load_broad, collect_broad
import pandas as pd

# 載入已採集的廣域新聞（若無資料，設 recollect=True 重新採集）
DAYS = 90          # 分析最近幾天
RECOLLECT = False  # 強制重新採集？

if RECOLLECT:
    df_broad = collect_broad(days=DAYS, save=True)
else:
    df_broad = load_broad(days=DAYS)

print(f'廣域新聞：{len(df_broad)} 篇')
print(f'日期範圍：{df_broad["published"].min().date()} ~ {df_broad["published"].max().date()}')
print(f'\n來源分布：')
print(df_broad['source'].value_counts().to_string())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['Arial Unicode MS', 'sans-serif']

# 每週文章數趨勢
weekly = df_broad.set_index('published').resample('W')['title'].count()
weekly.plot(kind='bar', figsize=(12, 3), title='廣域新聞：每週文章數', color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 看幾個標題樣本
print('\n標題樣本（前 10 篇）：')
for t in df_broad['title'].head(10):
    print(' >', t[:80])

## 2. BERTopic 自動題材發現

- **Embedding**：原始標題 → `paraphrase-multilingual-MiniLM-L12-v2`（保留語意）
- **關鍵詞**：jieba 斷詞 + 停用詞過濾 → c-TF-IDF（過濾媒體名稱）
- **聚類**：UMAP + HDBSCAN（自動決定 cluster 數）

⏱ 首次執行約 30-60 秒（Sentence Transformer 推理）

In [ ]:
from src.analyzers.topic_discovery import run_discovery

# 執行題材發現
result = run_discovery(
    days=DAYS,
    n_topics='auto',      # 讓 HDBSCAN 自動決定
    min_topic_size=8,     # 每個 cluster 最少文章數
    recollect=RECOLLECT,
)

print(f'\n發現 {result.n_topics} 個候選題材（共 {result.n_docs} 篇文章）')

## 3. 候選題材清單（人工審閱）

以下是自動發現的候選題材，每個 cluster 顯示：
- **關鍵詞**：該 cluster 的代表性詞彙
- **文章數**、**近週篇數**
- **樣本標題**：幫助判斷是否為真正的投資題材

In [ ]:
# 完整清單（可調整 top_n）
result.show(top_n=20)

In [ ]:
# 詳細查看某個候選題材的所有文章
INSPECT_RANK = 1  # 查看第幾名（從 1 開始）

if INSPECT_RANK <= len(result.candidates):
    c = result.candidates[INSPECT_RANK - 1]
    print(f'\n=== #{INSPECT_RANK} [{c.suggested_name}] ===')
    print(f'文章數：{c.doc_count}  關鍵詞：{c.keywords}')
    print('\n所有標題：')
    mask = result.df['_topic'] == c.topic_id
    for title in result.df.loc[mask, 'title']:
        print(' >', title[:100])

In [ ]:
# 繪製各候選題材的週頻趨勢（找有加速成長的題材）
import matplotlib.pyplot as plt
import math

n = min(len(result.candidates), 12)
cols = 3
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 2.5))
axes = axes.flatten()

for i, c in enumerate(result.candidates[:n]):
    ax = axes[i]
    c.weekly_trend.plot(ax=ax, color='royalblue', linewidth=1.5, marker='o', markersize=3)
    ax.set_title(f'#{i+1} {c.suggested_name[:20]}\n({c.doc_count} 篇)', fontsize=8)
    ax.set_xlabel('')
    ax.tick_params(axis='x', labelrotation=45, labelsize=6)
    ax.tick_params(axis='y', labelsize=7)
    ax.grid(True, alpha=0.3)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('各候選題材：週頻文章數趨勢', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

## 4. BERTopic 可視化

（選用）用 BERTopic 內建的互動式圖表探索各 cluster 的分布。

In [ ]:
# Topic 分布條形圖
try:
    fig = result.topic_model.visualize_barchart(top_n_topics=min(result.n_topics, 12), n_words=6)
    fig.show()
except Exception as e:
    print(f'視覺化需要 plotly：pip install plotly\n({e})')

In [ ]:
# 文件 2D 分布圖（需要 plotly）
try:
    docs_for_viz = result.df['title'].fillna('').tolist()
    fig = result.topic_model.visualize_documents(
        docs_for_viz,
        topics=result.df['_topic'].tolist(),
        hide_annotations=True,
        width=900, height=600,
    )
    fig.show()
except Exception as e:
    print(f'2D 分布需要 plotly：pip install plotly\n({e})')

## 5. 選取要加入監控的題材

看完上方的候選清單後，選出真正有投資意義的題材，
產生 `topics.yaml` 草稿，再人工補充股票與英文關鍵字。

In [ ]:
# === 在此選出你認為有意義的候選題材（依排名，從 1 開始）===
SELECTED_RANKS = [2, 3, 9, 10, 11]   # 例：第 2、3、9、10、11 名

# 印出選定的摘要
print('已選定的候選題材：')
for rank in SELECTED_RANKS:
    if rank <= len(result.candidates):
        c = result.candidates[rank - 1]
        print(f'  #{rank}: [{c.suggested_name}]  {c.doc_count} 篇')
        print(f'       關鍵詞：{c.keywords[:6]}')

In [ ]:
# 產生 YAML 草稿（只針對選定的題材）
import re

selected_candidates = [result.candidates[r-1] for r in SELECTED_RANKS if r <= len(result.candidates)]

lines = ["# ⚠️ BERTopic 自動發現候選題材 — 請人工填入 related_stocks 與 english keywords"]
lines += ["# 確認無誤後可複製貼入 config/topics.yaml\n"]

for c in selected_candidates:
    safe_name = re.sub(r"[^a-z0-9_]", "_",
                       c.suggested_name.lower().replace(" ", "_").replace("/", "_"))
    safe_name = re.sub(r"_+", "_", safe_name).strip("_") or f"topic_{c.topic_id}"
    lines += [
        f"# 自動發現 #{c.topic_id}  文章數={c.doc_count}  關鍵詞：{c.keywords[:5]}",
        f"{safe_name}:",
        f'  display_name: "{c.suggested_name}"  # ← 請改成有意義的中文名稱',
        f"  keywords:",
        f"    primary: [\"{c.keywords[0] if c.keywords else ''}\"]  # ← 請確認",
        f"    chinese: {c.keywords[:4]}  # ← 請補充/修正",
        f"    english: []   # ← 請補充英文關鍵字",
        f"  exclude_terms: []",
        f"  related_stocks:",
        f"    primary: []   # ← 請填入相關概念股 ticker（如 {{ticker: '2330.TW', name: '台積電', role: '...'}}）",
        f"    secondary: []",
        f"  time_range:",
        f"    start: \"2026-01-01\"",
        f"    end: \"2026-12-31\"",
        f"  benchmark: \"^TWII\"\n",
    ]

yaml_draft = "\n".join(lines)
print(yaml_draft)

In [ ]:
# （選用）存成暫存檔方便貼到 topics.yaml
out_path = ROOT / 'data' / 'raw' / 'news_broad' / 'topic_candidates_draft.yaml'
out_path.write_text(yaml_draft, encoding='utf-8')
print(f'草稿已存至：{out_path}')

## 6. 重新採集更多廣域新聞（可選）

若目前資料量不足（< 1000 篇），可以增加採集範圍或新增 RSS 來源。

In [ ]:
# 查看目前廣域新聞資料的完整狀況
from pathlib import Path
broad_dir = ROOT / 'data' / 'raw' / 'news_broad'
files = sorted(broad_dir.glob('broad_*.parquet'))
print(f'廣域新聞資料檔案：{len(files)} 個')
for f in files:
    df_f = __import__('pandas').read_parquet(f)
    print(f'  {f.name}: {len(df_f)} 篇')

print(f'\n目前總量：{len(df_broad)} 篇（{DAYS} 天範圍）')
print(f'建議：若 < 500 篇，可執行 collect_broad(days=180) 擴大採集範圍')

## 附錄：系統設計說明

```
廣域新聞採集 (news_broad.py)
    ↓
BERTopic 聚類 (topic_discovery.py)
    ├─ Embedding: paraphrase-multilingual-MiniLM-L12-v2（原始標題）
    ├─ UMAP 降維 → HDBSCAN 聚類
    └─ c-TF-IDF 關鍵詞（jieba 斷詞後文本）
    ↓
候選題材清單 (DiscoveryResult)
    ↓
人工審閱（本 notebook）
    ↓
config/topics.yaml（人工確認後加入）
    ↓
run_pipeline.py（自動化監控）
```

**設計原則**：系統提供候選清單，人仍是「守門員」。
自動發現的題材需人工確認關鍵字與成份股後，才加入正式監控清單。